# 02 - Feature Engineering

This notebook prepares the modelling-ready DataFrame (`df_model.csv`) from the wide-format data produced in notebook 01.

Steps:
1. Drop derived and helper columns
2. Multi-hot encode the `Species` column into binary component indicators
3. Standardise image paths to a consistent relative format
4. Save the final DataFrame to `data/processed/df_model.csv`

In [ ]:
import pandas as pd
from pathlib import Path

root_dir = Path("..")
processed_dir = root_dir / "data" / "processed"

## 1. Load wide-format data

Load `df_wide.csv` produced in notebook 01. This is the pivoted, quality-checked DataFrame with one row per image (357 rows x 11 columns before feature engineering).

In [ ]:
df_wide = pd.read_csv(processed_dir / "df_wide.csv")

## 2. Drop derived and helper columns

`Dry_Total_g` is a linear sum of the three biomass targets (`Dry_Clover_g + Dry_Dead_g + Dry_Green_g`). Including it as a feature would cause data leakage, and including it as a target is redundant -- total biomass will be derived from predictions at inference time.

The three helper columns (`sum_check`, `total_match`, `difference`) were used in EDA to verify this relationship and serve no modelling purpose.

`Sampling_Date` is also dropped here as it is redundant given the `Month` and `Season` features already present in `df_wide`.

In [ ]:
df_wide = df_wide.drop(columns=[
    "Dry_Total_g",       # target leakage -- derived from the other three targets
    "sum_check",         # EDA helper
    "total_match",       # EDA helper
    "difference",        # EDA helper
    "Sampling_Date",     # redundant -- Month and Season already capture temporal info
])

## 3. Multi-hot encode Species

The `Species` column contains underscore-separated compositions (e.g. `Phalaris_Clover_Ryegrass`). Treating this as a single categorical label would fail to capture shared species components across different compositions.

`str.get_dummies(sep="_")` splits each value on underscores and creates a binary indicator column per unique component. This produces one column per species found anywhere in the dataset. The original `Species` column is then dropped -- its information is fully captured by the new binary columns.

In [ ]:
species_dummies = df_wide["Species"].str.get_dummies(sep="_")
df_wide = pd.concat([df_wide, species_dummies], axis=1).drop(columns=["Species"])

## 4. Verify intermediate result

Quick sanity check before touching image paths.

In [ ]:
df_wide.head()

,image_path,Sampling_Date,State,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,GDM_g,Month,...,Fescue,Lucerne,Mixed,Phalaris,Ryegrass,SilverGrass,SpearGrass,SubcloverDalkeith,SubcloverLosa,WhiteClover
0,train/ID1011485656.jpg,2015-09-04,Tas,0.62,4.6667,0.0000,31.9984,16.2751,16.2750,9,...,0,0,0,0,1,0,0,0,0,0
1,train/ID1012260530.jpg,2015-04-01,NSW,0.55,16.0000,0.0000,0.0000,7.6000,7.6000,4,...,0,1,0,0,0,0,0,0,0,0
2,train/ID1025234388.jpg,2015-09-01,WA,0.38,1.0000,6.0500,0.0000,0.0000,6.0500,9,...,0,0,0,0,0,0,0,1,0,0
3,train/ID1028611175.jpg,2015-05-18,Tas,0.66,5.0000,0.0000,30.9703,24.2376,24.2376,5,...,0,0,0,0,1,0,0,0,0,0
4,train/ID1035947949.jpg,2015-09-11,Tas,0.54,3.5000,0.4343,23.2239,10.5261,10.9605,9,...,0,0,0,0,1,0,0,0,0,0


## 5. Standardise image paths

Image paths are stored as relative strings (`train/<filename>.jpg`). The cell below normalises them to ensure a consistent format regardless of the original source -- for example, it strips any leading directory components and rebuilds the path as `train/<stem>.jpg`.

AutoGluon expects absolute paths at training time. The conversion from relative to absolute will be handled in notebook 03 using the project root directory.

In [ ]:
# Inspect current path format before standardisation
df_wide["image_path"].head()

In [ ]:
# Rebuild path as "train/<filename>" -- idempotent on already-correct paths
df_wide["image_path"] = df_wide["image_path"].apply(
    lambda x: "train/" + Path(x).name
)

## 6. Save modelling DataFrame

Save the final feature-engineered DataFrame as `df_model.csv`. This is the file that will be loaded directly in notebook 03 for training.

In [ ]:
df_wide.to_csv(processed_dir / "df_model.csv", index=False)
print(f"Saved df_model.csv  --  shape: {df_wide.shape}")
df_wide.head()

In [ ]:
print(f"Columns ({df_wide.shape[1]} total):")
print(df_wide.columns.tolist())